# Gravitino model example

In [2]:
%pip install apache-gravitino==1.0.0 minio==7.2.16

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 1.4 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


### Show metalake, catalog

In [18]:
from typing import Dict, List
from gravitino import GravitinoMetalake, GravitinoAdminClient, GravitinoClient, NameIdentifier

# Create Gravitino admin client
metalake_name = "metalake_model"
catalog_name = "catalog_model"

gravitino_admin_client = GravitinoAdminClient(uri="http://gravitino:8090")
metalake_list: List[GravitinoMetalake] = gravitino_admin_client.list_metalakes()
print(metalake_list)

[GravitinoMetalake(_name='metalake_model', _comment='', _properties={'in-use': 'true'}, _audit=AuditDTO(_creator='anonymous', _create_time='2025-09-29T04:19:39.533751961Z', _last_modifier='anonymous', _last_modified_time='2025-09-29T04:19:49.544027052Z'))]


In [2]:
gravitino_client = GravitinoClient(uri="http://gravitino:8090", metalake_name=metalake_name)
catalog = gravitino_client.load_catalog(name=catalog_name)
print(catalog)

GenericModelCatalog(_name='catalog_model', _type=<Type.MODEL: ('model', True)>, _provider='model', _comment='catalog for ml models', _properties={'framework': 'tensorflow', 'in-use': 'true', 'algorithm': 'dense_nn'}, _audit=AuditDTO(_creator='anonymous', _create_time='2025-09-29T04:21:47.572217132Z', _last_modifier='anonymous', _last_modified_time='2025-09-29T04:45:53.087856679Z'))


### Create schema classfication

In [16]:
schema_name = "classification_models"
schema_path = "s3://ml-models/classification_models"
catalog.as_schemas().create_schema(
    schema_name=schema_name,
    comment="schema for classification models",
    properties={
        "localtion": schema_path,
        "task_type": "classification",
        "dataset": "synthetic"
    }
)

SchemaDTO(_name='classification_models', _comment='schema for classification models', _properties={'localtion': 's3://ml-models/classification_models', 'task_type': 'classification', 'dataset': 'synthetic'}, _audit=AuditDTO(_creator='anonymous', _create_time='2025-09-29T07:10:16.695848799Z', _last_modifier=None, _last_modified_time=None))

### Config Minio

In [7]:
from minio import Minio

MINIO_ENDPOINT = "minio:9000"
ACCESS_KEY = "minioadmin"
SECRET_KEY = "minioadmin"
BUCKET_NAME = "ml-models"

client = Minio(
    MINIO_ENDPOINT,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    secure=False
)

### Compute size of model

In [29]:
stat = client.stat_object("ml-models", "classification_models/model_1/saved_model_1_v0.zip")
size_bytes_v0 = stat.size
print("Model size:", size_bytes_v0)

stat = client.stat_object("ml-models", "classification_models/model_1/saved_model_1_v1.zip")
size_bytes_v1 = stat.size
print("Model size:", size_bytes_v1)

Model size: 758073
Model size: 759741


### Register model

In [31]:
model_ident = NameIdentifier.of("classification_models", "demo_classifier")
model_path = "s3://ml-models/classification_models/model_1"
catalog.as_schemas().register_model(
    ident=model_ident,
    comment="Demo classification model",  
    properties={
        "algorithm": "dense_nn",
        "framework": "tensorflow",
        "location": model_path
    }
)

### Link version

In [33]:
# Link version 1.0
catalog.link_model_version(
    model_ident = model_ident,
    uri="s3://ml-models/classification-models/model_1/saved_model_1_v0.zip",
    aliases=["v1.0", "baseline"],
    comment="Baseline model with basic parameters",
    properties={
        "model_size_bytes": size_bytes_v0
    }
)
  
# Link version 1.1  
catalog.link_model_version(
    model_ident = model_ident,
    uri="s3://ml-models/classification-models/model_1/saved_model_1_v1.zip",
    aliases=["v1.1", "production", "latest"],  
    comment="Improved model with better parameters",  
    properties={
        "model_size_bytes": size_bytes_v1,  
        "improvements": "increased_trees_and_depth"  
    }  
)

### Show version

In [34]:
versions = catalog.list_model_versions(model_ident)  
print(f"Available versions: {versions}")

Available versions: [0, 1]


### Get version by tag

In [35]:
prod_version = catalog.get_model_version_by_alias(model_ident, "production")
print(f"Production version: {prod_version.version()}")
print(f"Production accuracy: {prod_version.properties().get('model_size_bytes')}")

Production version: 1
Production accuracy: 759741
